# Write Results to Amazon RDS PostgreSQL

In [ ]:
## Setup: Load RDS Credentials from .env
# dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet') ## <<<
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet') ## <<<

from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import awswrangler as wr
import os
import pg8000

# Load environment variables from .env file
dotenv_path = f'{project_src}/.env'
load_dotenv(dotenv_path)

# Get credentials from environment variables
RDS_ENDPOINT = os.getenv('RDS_ENDPOINT')
RDS_PORT = os.getenv('RDS_PORT', '5432')
RDS_DATABASE = os.getenv('RDS_DATABASE')
RDS_USERNAME = os.getenv('RDS_USERNAME')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')

# Validate that all credentials are present
if not all([RDS_ENDPOINT, RDS_DATABASE, RDS_USERNAME, RDS_PASSWORD]):
    raise ValueError("❌ Missing RDS credentials in .env file. Please fill in all values.")

# Create connection string
db_connection_string = f"postgresql://{RDS_USERNAME}:{RDS_PASSWORD}@{RDS_ENDPOINT}:{RDS_PORT}/{RDS_DATABASE}"

try:
    engine = pg8000.connect(
                            host=RDS_ENDPOINT,
                            database=RDS_DATABASE,
                            user=RDS_USERNAME,
                            password=RDS_PASSWORD,
                            port=RDS_PORT
                            )
    # Test connection
except Exception as e:
    print(f"✗ Connection failed: {e}")

In [ ]:
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
print(f"Shape of dim_person: {dim_person.shape}")
dim_person = dim_person.head(100)
print(f"Shape of dim_person (sub_sample): {dim_person.shape}")

Shape of dim_person: (10777803, 6)
Shape of dim_person (sub_sample): (100, 6)


In [ ]:
## Helper Function: Write DataFrame to RDS

def write_dataframe_to_rds(df, table_name, if_exists='append'):
    """
    Write a pandas DataFrame to RDS PostgreSQL
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to write
    table_name : str
        Name of the table in RDS (e.g., 'strongest_collaborations')
    if_exists : str
        How to behave if table exists:
        - 'fail': Raise an error (default pandas behavior)
        - 'replace': Drop table and create new one
        - 'append': Insert data into existing table
    
    Returns:
    --------
    bool : True if successful, False if failed
    """
    try:
        # Convert table name to lowercase (PostgreSQL convention)
        table_name = table_name.lower()
        
        # Write to database
        # Write to RDS (Wrangler handles the chunking and optimal copy commands)
        wr.postgresql.to_sql(
            df=df,
            table=table_name,
            schema="public",
            con=engine,
            mode=if_exists, # or "overwrite"
            use_column_names=True
        )
        print(f"✓ Successfully wrote {len(df)} rows to table '{table_name}' (mode: {if_exists})")
        return True
    except Exception as e:
        print(f"✗ Failed to write to table '{table_name}': {e}")
        return False

In [ ]:
## Write All Analysis Results to RDS

# Write each analysis result to RDS
# Uncomment the ones you want to write, or modify table names as needed

print("=" * 60)
print("WRITING ANALYSIS RESULTS TO RDS")
print("=" * 60)

# Analysis 1: Most Active Participants
try:
    dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
    dim_person = dim_person.head(100000)
    write_dataframe_to_rds(dim_person, 'dim_person', if_exists='overwrite')
except Exception as e:
    print(f"⚠ Could not load dim_person: {e}")


print("\n" + "=" * 60)
print("WRITE PROCESS COMPLETE")
print("=" * 60)
engine.close()

WRITING ANALYSIS RESULTS TO RDS
✓ Successfully wrote 100000 rows to table 'dim_person' (mode: overwrite)

WRITE PROCESS COMPLETE


In [ ]:
## Custom: Write Any DataFrame to RDS

# Use this cell to write any dataframe you want to RDS
# Replace the variable names and table name with your values

# Example template:
# ==================

# 1. Define your dataframe variable (it should already exist in memory)
# my_dataframe = <your_dataframe_variable>

# 2. Call the helper function
# write_dataframe_to_rds(my_dataframe, 'your_table_name', if_exists='append')

# ==================

# Example: Write your custom analysis
# write_dataframe_to_rds(your_df, 'your_custom_table', if_exists='append')

In [ ]:
## Verification: Check What's in RDS

# Get list of all tables in the database
inspector = inspect(engine)
all_tables = inspector.get_table_names()

print("Tables in RDS database:")
print("=" * 60)
for i, table in enumerate(all_tables, 1):
    # Get row count for each table
    row_count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", con=engine)['count'].values[0]
    print(f"{i}. {table}: {row_count:,} rows")

print("\n" + "=" * 60)
print(f"Total tables: {len(all_tables)}")

# Optional: Read sample data from a specific table
# Replace 'table_name' with the actual table name
# query = "SELECT * FROM table_name LIMIT 5"
# sample = pd.read_sql(query, con=engine)
# print(sample)

NoInspectionAvailable: No inspection system is available for object of type <class 'pg8000.legacy.Connection'>